source : https://stackoverflow.com/questions/62555987/lightgbm-ranking-example

In [40]:
# Source - https://stackoverflow.com/a/67621253
# Posted by charelf, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-03, License - CC BY-SA 4.0

import numpy as np
import pandas as pd
import catboost
from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import GroupShuffleSplit

In [41]:
def load_letor_file(file_path):
    X, y = load_svmlight_file(file_path)

    X.indices = X.indices.astype(np.int32)
    X.indptr = X.indptr.astype(np.int32)
    X.sort_indices() # Ensures internal array consistency

    # Extract the query IDs (qid:XXX) to build the group array
    with open(file_path, 'r') as f:
        print()
        qids = [int(line.split(" ")[1].split(':')[1]) for line in f]
    print("qids : ",qids)

    # Count documents per query (maintaining original order)
    # _, counts = np.unique(qids, return_counts=True)
    return X, y, qids

In [42]:
ranker = catboost.CatBoostRanker(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='YetiRankPairwise',
    eval_metric='NDCG'
)

In [43]:
gss = GroupShuffleSplit(n_splits=1, train_size=0.8)

In [44]:
# Define paths to files based on your directory structure
# Example assumes you have files like "Fold1/train.txt", "Fold1/vali.txt", "Fold1/test.txt"
train_path = "./ranking_dataset_merged.txt"      # Contains S1, S2, S3 combined


In [45]:
# 1. Load data
X, y, q = load_letor_file(train_path)

train_idx, test_idx = next(gss.split(X, y, groups=q))

X_train = X[train_idx]
y_train = y[train_idx]
q_train = [q[i] for i in train_idx]

X_test = X[test_idx]
y_test = y[test_idx]
q_test = [q[i] for i in test_idx]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_data = catboost.Pool(data=X_train, label=y_train, group_id=q_train)
test_data = catboost.Pool(data=X_test, label=y_test, group_id=q_test)


qids :  [1001, 1001, 1001, 1001, 1001, 1001, 1001, 1001, 1001, 1001, 1001, 1002, 1002, 1002, 1002, 1002, 1002, 1002, 1002, 1002, 1002, 1002, 1003, 1003, 1003, 1003, 1003, 1003, 1003, 1003, 1003, 1003, 1003, 1004, 1004, 1004, 1004, 1004, 1004, 1004, 1004, 1004, 1004, 1004, 1005, 1005, 1005, 1005, 1005, 1005, 1005, 1005, 1005, 1005, 1005, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1007, 1007, 1007, 1007, 1007, 1007, 1007, 1007, 1008, 1008, 1008, 1008, 1008, 1008, 1008, 1008, 1008, 1008, 1008, 1009, 1009, 1009, 1009, 1009, 1009, 1009, 1009, 1009, 1009, 1009, 1009, 10010, 10010, 10010, 10010, 10010, 10010, 10010, 10010, 10010, 10010, 10010, 10011, 10011, 10011, 10011, 10011, 10011, 10011, 10011, 10011, 10011, 10011, 10011, 10012, 10012, 10012, 10012, 10012, 10012, 10012, 10012, 10012, 10012, 10012, 10013, 10013, 10013, 10013, 10013, 10013, 10013, 10013, 10013, 10013, 10014, 10014, 10014, 10014, 10014, 10014, 10014, 10014, 10014, 10014, 10014, 10014, 10015, 10015, 10

In [46]:
ranker.fit(train_data, eval_set=test_data)

0:	test: 0.9230433	best: 0.9230433 (0)	total: 9.19ms	remaining: 9.18s
1:	test: 0.9419588	best: 0.9419588 (1)	total: 16.3ms	remaining: 8.12s
2:	test: 0.9445328	best: 0.9445328 (2)	total: 23.5ms	remaining: 7.81s
3:	test: 0.9422446	best: 0.9445328 (2)	total: 28.9ms	remaining: 7.21s
4:	test: 0.9422446	best: 0.9445328 (2)	total: 34.9ms	remaining: 6.95s
5:	test: 0.9302589	best: 0.9445328 (2)	total: 41.3ms	remaining: 6.84s
6:	test: 0.9294472	best: 0.9445328 (2)	total: 45.4ms	remaining: 6.44s
7:	test: 0.9294472	best: 0.9445328 (2)	total: 51.1ms	remaining: 6.33s
8:	test: 0.9344645	best: 0.9445328 (2)	total: 56ms	remaining: 6.17s
9:	test: 0.9336528	best: 0.9445328 (2)	total: 60.3ms	remaining: 5.97s
10:	test: 0.9336528	best: 0.9445328 (2)	total: 64.8ms	remaining: 5.82s
11:	test: 0.9364667	best: 0.9445328 (2)	total: 68.9ms	remaining: 5.67s
12:	test: 0.9356599	best: 0.9445328 (2)	total: 72.6ms	remaining: 5.51s
13:	test: 0.9369463	best: 0.9445328 (2)	total: 76.1ms	remaining: 5.36s
14:	test: 0.934403

CatBoostRanker(depth=6, eval_metric='NDCG', iterations=1000, learning_rate=0.1, loss_function='YetiRankPairwise')

In [47]:
# Get predictions and print rankings
predictions = ranker.predict(X_train)
print("Final Predictions (Rankings):")
print(predictions[:20])  # Print first 20 predictions

Final Predictions (Rankings):
[ 1.00156197  2.17996015  1.23235912  2.08413654 -0.1480121   0.35889728
  0.27732608 -2.14861301 -2.14025249 -2.10569091 -2.14456375  1.57890878
  2.83973255  1.71695754  0.50186453  0.50186453  0.1918576   0.1918576
 -1.0249153  -2.15828402]


In [28]:
# Save the model
ranker.save_model('catboost_ranker_model.txt')
print("Model saved to 'catboost_ranker_model.txt'")


Model saved to 'catboost_ranker_model.txt'


In [50]:
# Load the model for reuse
new_ranker = catboost.CatBoostRanker() 
new_ranker.load_model('catboost_ranker_model.txt')
print("Model loaded from 'catboost_ranker_model.txt'")

# data, _, _ = load_letor_file("./ranking_dataset.txt")

# test_data = data[0:12]
test_data = np.array([
    [0.7157, 1.0000, 0.8000, 0.8000, 0.0000, 0.0000, 0.5000],
    [0.7883, 0.6667, 0.8000, 0.8000, 0.0000, 0.0000, 0.5000],
    [0.7137, 0.0000, 0.8333, 0.8000, 0.0000, 1.0000, 0.5000],
    [0.7117, 0.6667, 0.8000, 0.8000, 0.0000, 0.0000, 0.5000],
    [0.7098, 0.3333, 0.8000, 0.8000, 0.0000, 1.0000, 0.5000],
    [0.1815, 0.2362, 0.7019, 0.7414, 0.0000, 0.1723, 0.3853],
    [0.8206, 0.8563, 0.1417, 0.1074, 1.0000, 0.8605, 0.7797],
    [0.0177, 0.1997, 0.3464, 0.7976, 1.0000, 0.5708, 0.9363],
    [0.3384, 0.5887, 0.7072, 0.6747, 0.0000, 0.5434, 0.4048]
])
print(test_data)

# Get rankings for new data
new_predictions = new_ranker.predict(test_data)
print("\nNew predictions:")
print(new_predictions)

# Sort items by ranking score (highest first)
ranked_indices = np.argsort(-new_predictions)  # Negative for descending order
print("\nRanked indices (best to worst):")
print(ranked_indices)
print("\nRanked scores:")
print(new_predictions[ranked_indices])


Model loaded from 'catboost_ranker_model.txt'
[[0.7157 1.     0.8    0.8    0.     0.     0.5   ]
 [0.7883 0.6667 0.8    0.8    0.     0.     0.5   ]
 [0.7137 0.     0.8333 0.8    0.     1.     0.5   ]
 [0.7117 0.6667 0.8    0.8    0.     0.     0.5   ]
 [0.7098 0.3333 0.8    0.8    0.     1.     0.5   ]
 [0.1815 0.2362 0.7019 0.7414 0.     0.1723 0.3853]
 [0.8206 0.8563 0.1417 0.1074 1.     0.8605 0.7797]
 [0.0177 0.1997 0.3464 0.7976 1.     0.5708 0.9363]
 [0.3384 0.5887 0.7072 0.6747 0.     0.5434 0.4048]]

New predictions:
[ 0.27015055  0.2665775   0.07794939  0.24053988  0.09901781 -0.39944633
 -0.40101522 -0.36883279 -0.39127223]

Ranked indices (best to worst):
[0 1 3 4 2 7 8 5 6]

Ranked scores:
[ 0.27015055  0.2665775   0.24053988  0.09901781  0.07794939 -0.36883279
 -0.39127223 -0.39944633 -0.40101522]
